# PC biplots -- round 2

Same SPLOM as `pc_biplots.ipynb`, sourced from `submit_pca_r2.ipynb`'s output instead of round 1's. Round 2's reference is the full EUR pool (`PCA_REF_POPS` in that notebook: CEU/GBR/TSI/FIN/IBS), not just CEU/GBR -- each of those 5 populations gets its own distinguishable shade of blue plus its own marker shape from the shared superpopulation palette (same generation as `round1_filter.ipynb`/`round2_filter.ipynb`/`pc_biplots.ipynb`), so the EUR sub-structure round 2 is meant to resolve is actually visible here, not lumped into one color.

**Performance:** same handling as `pc_biplots.ipynb` -- AoU is randomly subsampled to `MAX_AOU_POINTS`, every scatter layer is `rasterized=True`.

In [ ]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

## Inputs

In [ ]:
WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
BUCKET_DIR = f"{WORKSPACE_BUCKET}/data/01_ancestry_filtering"
R2_OUT_DIR = f"{BUCKET_DIR}/round2_pca"

CLUSTER = "ceu_gbr"   # matches submit_pca_r2.ipynb's CLUSTER

REF_PCS_PATH = f"{R2_OUT_DIR}/r2_pca_{CLUSTER}_reprojected.sscore"
AOU_PCS_PATH = f"{R2_OUT_DIR}/r2_projected_{CLUSTER}.sscore"
REF_PANEL_PATH = f"{BUCKET_DIR}/integrated_call_samples_v3.20130502.ALL.panel"
OUT_DIR = R2_OUT_DIR   # figure written here

N_PCS_TO_PLOT = 6       # panel grid is N_PCS_TO_PLOT^2 -- grows fast, keep this modest
MAX_AOU_POINTS = 20000  # random subsample cap for plotting only, not for any analysis
RANDOM_SEED = 0

os.makedirs(OUT_DIR, exist_ok=True)

## Load reference PCs + population labels, build the population palette

Same superpopulation-grouped palette as `round1_filter.ipynb`/`round2_filter.ipynb`/`pc_biplots.ipynb` -- one colormap per continent, distinguishable shades within it, plus a marker shape per population. Round 2's reference only ever contains EUR populations, but built from the full 26-population palette anyway for consistency (and in case that changes).

In [ ]:
pc_cols = [f"PC{i}_AVG" for i in range(1, N_PCS_TO_PLOT + 1)]

ref = pd.read_csv(REF_PCS_PATH, sep=r"\s+")
ref_id_col = "#IID" if "#IID" in ref.columns else "IID"
ref = ref.rename(columns={ref_id_col: "sample"})

labels = pd.read_csv(REF_PANEL_PATH, sep="\t")
ref = ref.merge(labels[["sample", "pop", "super_pop"]], on="sample")
assert all(c in ref.columns for c in pc_cols), f"missing PC columns, have: {list(ref.columns)}"

SUPERPOP_POPS = {
    "EUR": ["CEU", "TSI", "FIN", "GBR", "IBS"],
    "AFR": ["YRI", "LWK", "GWD", "MSL", "ESN", "ASW", "ACB"],
    "EAS": ["CHB", "JPT", "CHS", "CDX", "KHV"],
    "SAS": ["GIH", "PJL", "BEB", "STU", "ITU"],
    "AMR": ["MXL", "PUR", "CLM", "PEL"],
}
SUPERPOP_CMAPS = {"EUR": "Blues", "AFR": "Oranges", "EAS": "Greens", "SAS": "Purples", "AMR": "Reds"}

# varied per superpopulation rather than one fixed (lo, hi) for everyone -- keeps shades
# from converging toward similar mid-tones across different continents
SUPERPOP_SHADE_RANGES = {
    "EUR": (0.25, 0.95),
    "AFR": (0.35, 0.90),
    "EAS": (0.30, 0.95),
    "SAS": (0.25, 0.85),
    "AMR": (0.35, 0.90),
}

# cycled per population within each superpopulation -- a second, shape-based channel of
# differentiation on top of color
POP_MARKERS_CYCLE = ["o", "s", "^", "D", "v", "P", "X", "*"]


def build_pop_colors(superpop_pops=SUPERPOP_POPS, superpop_cmaps=SUPERPOP_CMAPS, shade_ranges=SUPERPOP_SHADE_RANGES):
    colors = {}
    for superpop, pops in superpop_pops.items():
        cmap = matplotlib.colormaps[superpop_cmaps[superpop]]
        lo, hi = shade_ranges[superpop]
        shades = np.linspace(lo, hi, len(pops)) if len(pops) > 1 else [(lo + hi) / 2]
        for pop, frac in zip(pops, shades):
            colors[pop] = mcolors.to_hex(cmap(frac))
    return colors


def build_pop_markers(superpop_pops=SUPERPOP_POPS, markers_cycle=POP_MARKERS_CYCLE):
    markers = {}
    for pops in superpop_pops.values():
        for i, pop in enumerate(pops):
            markers[pop] = markers_cycle[i % len(markers_cycle)]
    return markers


POP_COLORS = build_pop_colors()
POP_MARKERS = build_pop_markers()
print(f"Reference: {len(ref)} samples, populations: {sorted(ref['pop'].unique())}")

## Load + subsample AoU

Subsample happens once here, up front -- every panel in the grid reuses the same `aou_sub`, rather than each panel drawing its own random subset (which would make different panels show different, incomparable subsets of samples).

In [ ]:
t0 = time.time()
aou = pd.read_csv(AOU_PCS_PATH, sep=r"\s+")
aou_id_col = "#IID" if "#IID" in aou.columns else "IID"
aou = aou.rename(columns={aou_id_col: "sample"})
assert all(c in aou.columns for c in pc_cols), f"missing PC columns, have: {list(aou.columns)}"
print(f"AoU: {len(aou)} samples loaded in {time.time() - t0:.1f}s")

if len(aou) > MAX_AOU_POINTS:
    aou_sub = aou.sample(n=MAX_AOU_POINTS, random_state=RANDOM_SEED)
else:
    aou_sub = aou
print(f"Plotting {len(aou_sub)} AoU points (subsampled from {len(aou)})")

## Pairwise scatterplot matrix

Lower triangle only. Diagonal shows each PC's marginal distribution (reference vs AoU) instead of a scatter against itself. AoU drawn first (background, grey), reference drawn on top (foreground, full alpha) -- both layers rasterized regardless of point count. Each EUR population gets its own color and marker -- if round 2's tighter CEU/GBR ellipsoid is doing its job, CEU/GBR should visually separate from TSI/FIN/IBS here, and AoU's passing subset should cluster with CEU/GBR specifically rather than smearing across all five.

In [ ]:
t0 = time.time()
n = len(pc_cols)
fig, axes = plt.subplots(n, n, figsize=(2.2 * n, 2.2 * n))

for row in range(n):
    for col in range(n):
        ax = axes[row, col]
        if row < col:
            ax.axis("off")
            continue

        y_col, x_col = pc_cols[row], pc_cols[col]

        if row == col:
            ax.hist(ref[x_col], bins=40, density=True, color="grey", alpha=0.6, label="1000G")
            ax.hist(aou_sub[x_col], bins=40, density=True, color="royalblue", alpha=0.4, label="AoU")
            if row == 0:
                ax.legend(fontsize=6, loc="upper right")
        else:
            ax.scatter(
                aou_sub[x_col], aou_sub[y_col],
                c="grey", s=3, alpha=0.3, linewidths=0, rasterized=True,
            )
            for pop, grp in ref.groupby("pop"):
                ax.scatter(
                    grp[x_col], grp[y_col],
                    c=POP_COLORS.get(pop, "black"), marker=POP_MARKERS.get(pop, "o"),
                    s=6, alpha=1.0, linewidths=0, rasterized=True,
                )

        if row == n - 1:
            ax.set_xlabel(x_col, fontsize=8)
        else:
            ax.set_xticklabels([])
        if col == 0:
            ax.set_ylabel(y_col, fontsize=8)
        else:
            ax.set_yticklabels([])
        ax.tick_params(labelsize=6)

handles = [
    plt.Line2D([0], [0], marker=POP_MARKERS.get(pop, "o"), linestyle="", color=POP_COLORS[pop], label=pop)
    for pop in sorted(ref["pop"].unique())
]
handles.append(plt.Line2D([0], [0], marker="o", linestyle="", color="grey", label="AoU"))
fig.legend(handles=handles, loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=7, title="Population")

plt.tight_layout()
plot_path = os.path.join(OUT_DIR, f"pc_biplots_r2_{CLUSTER}.png")
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {plot_path} in {time.time() - t0:.1f}s")